# Linly-Dubbing Colab WebUI
## 1. 环境准备 (Environment Setup)
基于第一性原理，我们需要首先确认硬件环境，然后安装操作系统级别的底层依赖，最后才是应用层的Python库。

In [ ]:
# [Step 1.1] 硬件检查
# 确认分配到的 GPU 类型
!nvidia-smi

In [ ]:
# [Step 1.2] 获取代码
# 使用幂等性逻辑：如果目录已存在，则跳过 clone，避免重复执行报错
import os
if not os.path.exists('/content/Linly-Dubbing'):
    %cd /content/
    !git clone https://github.com/infinite-gaming-studio/Linly-Dubbing.git --depth 1
else:
    print("Project already cloned.")

%cd /content/Linly-Dubbing
!git submodule update --init --recursive

In [ ]:
# [Step 1.3] 安装系统级依赖 (System Dependencies)
# 优先安装系统库，确保 C++ 编译环境就绪
!apt-get update -qq
!apt-get install -y -qq build-essential libfst-dev ffmpeg espeak-ng libsndfile1 \
    libavfilter-dev libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev > /dev/null
print("System dependencies installed.")
!ffmpeg -version | head -n 1

In [ ]:
# [Step 1.4] Python 依赖安装 (Python Dependencies via uv)
# 我们使用 'uv' 进行极速安装。Colab 环境需要特定版本的库以保证 ASR/TTS 兼容性。

# 1. 安装 uv
!pip install -q uv

# 2. [CRITICAL] 准备构建环境 (Fix for pynini build failure)
print("Installing build dependencies (Cython)...")
!uv pip install --system Cython

# 3. [CRITICAL] 强制重装核心库 (Fix for numpy, setuptools, torch compatibility)
print("Installing core compute libraries... This might take a minute.")
# numpy<2.0.0 is required for many binary extensions
# torch==2.3.1 is required for compatibility with the current code base and fake tensor support
!uv pip install --system --force-reinstall "numpy<2.0.0" "setuptools" torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1

# 4. 运行时修补 (Runtime Patching)
# 将 numpy==1.26.3 修改为 numpy<2.0.0 以提高灵活性
!sed -i 's/numpy==1.26.3/numpy<2.0.0/g' requirements.txt

# [CRITICAL] Patch TTS submodule for Python 3.12 support (Colab uses 3.12)
!sed -i 's/if Version(python_version) < Version("3.9") or Version(python_version) >= Version("3.12"):/# if Version(python_version) < Version("3.9") or Version(python_version) >= Version("3.12"):/g' submodules/TTS/setup.py
!sed -i 's/    raise RuntimeError("TTS requires python >= 3.9 and < 3.12 " "but your Python version is {}".format(sys.version))/#     raise RuntimeError("TTS requires python >= 3.9 and < 3.12 " "but your Python version is {}".format(sys.version))/g' submodules/TTS/setup.py
!sed -i 's/python_requires=">=3.9.0, <3.12",/python_requires=">=3.9.0, <3.13",/g' submodules/TTS/setup.py
print("Patched dependencies for Python 3.12 compatibility.")

# 5. 安装项目及子模块依赖
print("Installing project requirements...")
!uv pip install --system -r requirements.txt
!uv pip install --system -r requirements_module.txt

print("Python dependencies installed successfully.")

In [ ]:
# [Step 1.6] Patch videotrans library for Colab (Headless Support)
# Fixes IndexError: list index out of range in set_process by mocking GUI components
import os
import site

def patch_videotrans_tools():
    # Find videotrans location in site-packages
    site_packages = site.getsitepackages()
    target_file = None
    
    for sp in site_packages:
        possible_path = os.path.join(sp, 'videotrans', 'util', 'tools.py')
        if os.path.exists(possible_path):
            target_file = possible_path
            break
            
    if not target_file:
        # Fallback: try to import and get file
        try:
            import videotrans.util.tools
            target_file = videotrans.util.tools.__file__
        except ImportError:
            print("Could not find videotrans to patch. It might not be installed yet.")
            return

    print(f"Patching {target_file}...")
    
    with open(target_file, 'r', encoding='utf-8') as f:
        content = f.read()

    patch_code = """

# --- COLAB PATCH START ---
# Mock UI components for headless execution
class DummyUI:
    def error(self, msg): print(f"[UI Error] {msg}")
    def info(self, msg): print(f"[UI Info] {msg}")
    def warning(self, msg): print(f"[UI Warning] {msg}")
    def log(self, msg): print(f"[UI Log] {msg}")
    def setText(self, text): print(f"[UI Status] {text}")
    def set_value(self, val): pass

# Ensure log_ui_ui exists and has elements
if 'log_ui_ui' not in globals() or not log_ui_ui:
    log_ui_ui = [DummyUI() for _ in range(5)]

# Override set_process to be safe
def set_process(text=None, step=None):
    try:
        if text:
            print(f"[Processing] {text}")
            if 'log_ui_ui' in globals() and len(log_ui_ui) > 0:
                log_ui_ui[0].setText(text)
        if step is not None:
             if 'log_ui_ui' in globals() and len(log_ui_ui) > 1:
                log_ui_ui[1].set_value(step)
    except Exception as e:
        print(f"Error in set_process: {e}")
# --- COLAB PATCH END ---
"""
    
    if "# --- COLAB PATCH START ---" in content:
        print("File already patched.")
    else:
        with open(target_file, 'a', encoding='utf-8') as f:
            f.write(patch_code)
        print("Patch applied successfully.")

try:
    patch_videotrans_tools()
except Exception as e:
    print(f"An error occurred while patching: {e}")


## 2. 资源获取 (Resource Acquisition)
下载运行所需的预训练模型文件。

In [ ]:
# [Step 2.1] 下载模型
# 创建目录结构
!mkdir -p models/ASR/whisper

# 下载 wav2vec2 模型 (使用 -nc 参数避免重复下载)
!wget -nc https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth \
    -O models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth

# 运行项目自带的下载脚本下载其他模型
!python scripts/huggingface_download.py

## 3. 应用启动 (Application Launch)
配置环境并启动 WebUI。

In [ ]:
# [Step 3.1] 环境配置
%cd /content/Linly-Dubbing
!cp env.example .env
# 如果需要配置 API Key，请手动编辑 .env 文件或在 WebUI 界面中设置

In [ ]:
# [Step 3.2] 功能自检 (可选)
# 测试核心功能模块是否正常加载
!python -m tools.do_everything

In [ ]:
# [Step 3.3] 启动 WebUI
# 启动后点击输出结果中的 public URL 即可访问
!python webui.py